# Conditional VAE Generation

Generate equal numbers for five minority label combinations. These files are used only in classifier training.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import torch
from torchvision.utils import save_image

sys.path.append(str(Path.cwd()))
from multilabel_utils import LABEL_COLUMNS, get_device, set_seed
from generative_models import ConditionalVAE

SEED = 42
LATENT_DIM = 64
SAMPLES_PER_COMBINATION = 100
set_seed(SEED)
device = get_device()
PROJECT_ROOT = Path.cwd().parents[1]
MODEL_PATH = PROJECT_ROOT / "models" / "multilabel" / "conditional_vae_best.pth"
OUTPUT_DIR = PROJECT_ROOT / "data" / "synthetic" / "multilabel_vae"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

conditions = {
    "delamination": [0, 1, 0],
    "pinhole": [0, 0, 1],
    "crack_delamination": [1, 1, 0],
    "crack_pinhole": [1, 0, 1],
    "all_three": [1, 1, 1],
}

In [2]:
model = ConditionalVAE(LATENT_DIM, len(LABEL_COLUMNS)).to(device)
model.load_state_dict(torch.load(MODEL_PATH, map_location=device, weights_only=True))
model.eval()

records = []
with torch.no_grad():
    for condition_name, condition_values in conditions.items():
        condition = torch.tensor(
            condition_values, dtype=torch.float32, device=device
        ).repeat(SAMPLES_PER_COMBINATION, 1)
        latent_vectors = torch.randn(
            SAMPLES_PER_COMBINATION, LATENT_DIM, device=device
        )
        images = model.decode(latent_vectors, condition).cpu()

        for index, image in enumerate(images):
            filename = f"vae_{condition_name}_{index:04d}.png"
            save_image(image, OUTPUT_DIR / filename)
            records.append({
                "file_name": filename,
                "image_path": str(Path("data") / "synthetic" / "multilabel_vae" / filename),
                "Surface_Crack": condition_values[0],
                "Delamination": condition_values[1],
                "Pinhole": condition_values[2],
                "is_synthetic": True,
            })

metadata = pd.DataFrame(records)
metadata.to_csv(OUTPUT_DIR / "metadata.csv", index=False)
print("Generated images:", len(metadata))
print(metadata[LABEL_COLUMNS].sum())

Generated images: 500
Surface_Crack    300
Delamination     300
Pinhole          300
dtype: int64


Generated labels describe the condition requested from the model. They do not guarantee that every visual defect is actually present, so downstream results and sample quality must be interpreted together.